## Unified cell-data processor (frequency-sorted gene reordering)

Combines `data_processing.ipynb` (single-organ) and `data_process_v2.ipynb` (8 organs)
into one notebook. The only differences between those two notebooks are exposed as
variables in the **Configuration** cell below — edit them to switch behavior.

**Defaults reproduce `data_process_v2.ipynb`** (8 organs, metadata stripped, `*_sorted_noorgan.json`).

To reproduce `data_processing.ipynb`, set **all three** knobs together:
```python
ORGAN_PATTERNS    = [("heart", "heart_cell_data_part_*.json")]
STRIP_METADATA    = False
OUTPUT_NAME_STYLE = "stem"
```
(All three matter: the global gene-frequency table — and therefore the sort order — depends on
which files are loaded and whether metadata is stripped, not on the organ-list length alone.)


In [ ]:
import json
import re
from pathlib import Path
from collections import Counter

# ============================================================
# Configuration  --  EDIT THESE to switch behavior
# (raw inputs only; computed paths/names are derived in the body cells below)
# ============================================================
GENES_PREFIX = "Genes: "

DATA_DIR = "data"          # folder containing the source *_cell_data_part_*.json files
OUT_DIR  = "sorted_data"   # folder to write the frequency-sorted outputs into

# (organ_name, glob_pattern) pairs to process. A single entry = single-organ run.
# brain files have no organ prefix, so their pattern is the bare "cell_data_part_*.json".
ORGAN_PATTERNS = [
    ("bone-marrow", "bone-marrow_cell_data_part_*.json"),
    ("bladder",     "bladder_cell_data_part_*.json"),
    ("liver",       "liver_cell_data_part_*.json"),
    ("lung",        "lung_cell_data_part_*.json"),
    ("kidney",      "kidney_cell_data_part_*.json"),
    ("heart",       "heart_cell_data_part_*.json"),
    ("limb-muscle", "limb-muscle_cell_data_part_*.json"),
    ("brain",       "cell_data_part_*.json"),  # brain has no prefix
]

# Strip metadata lines (Gender/Class/Tissue), keeping only the "Genes:" line, before parsing.
STRIP_METADATA = True

# Output filename style:
#   "noorgan" -> {organ}_celldata_part_{num}_sorted_noorgan.json   (data_process_v2.ipynb)
#   "stem"    -> {stem}_sorted.json                                (data_processing.ipynb)
# "stem" is intended for single-organ runs; with a multi-organ ORGAN_PATTERNS it drops the
# organ label (brain outputs would be named from their bare stem).
OUTPUT_NAME_STYLE = "noorgan"
assert OUTPUT_NAME_STYLE in ("noorgan", "stem"), f"unknown OUTPUT_NAME_STYLE: {OUTPUT_NAME_STYLE!r}"
# "stem" carries no organ label, so it only makes sense for single-organ runs.
assert OUTPUT_NAME_STYLE != "stem" or len(ORGAN_PATTERNS) == 1, "stem naming has no organ label; use noorgan for multi-organ runs"

In [ ]:
# ============================================================
# Helpers
# ============================================================
def strip_metadata(input_str):
    """Remove metadata lines (e.g. '\nGender: male\nClass: ...\nTissue: ...'),
    keeping only the 'Genes: ...' line."""
    if not isinstance(input_str, str):
        return input_str
    if GENES_PREFIX not in input_str:
        return input_str
    genes_lines = [line for line in input_str.split("\n") if line.startswith(GENES_PREFIX)]
    return genes_lines[0] if genes_lines else input_str


def parse_genes(input_str):
    """Parse 'Genes: gene1 count1 gene2 count2 ...' -> list[(gene, count_str)]."""
    if not isinstance(input_str, str):
        return []
    if not input_str.startswith(GENES_PREFIX):
        return []
    tokens = input_str[len(GENES_PREFIX):].split()
    pairs = []
    for i in range(0, len(tokens) - 1, 2):  # safe for odd token counts
        pairs.append((tokens[i], tokens[i + 1]))
    return pairs


def prepare_input(input_str):
    """Apply metadata stripping only when STRIP_METADATA is enabled."""
    return strip_metadata(input_str) if STRIP_METADATA else input_str

In [ ]:
# ============================================================
# Collect source files from all configured organ patterns
# ============================================================
src_files = []  # list of (organ_name, Path)
for organ_name, pattern in ORGAN_PATTERNS:
    files = sorted(Path(DATA_DIR).glob(pattern))
    # Only take original inputs; skip already-sorted / classifier outputs.
    files = [p for p in files if "_sorted" not in p.stem and "_cls" not in p.stem]
    src_files.extend((organ_name, p) for p in files)

print(f"found {len(src_files)} source files across {len(ORGAN_PATTERNS)} pattern(s)")
for organ_name, _ in ORGAN_PATTERNS:
    count = sum(1 for o, _ in src_files if o == organ_name)
    print(f"  {organ_name}: {count} files")

out_dir = Path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
print(f"\nwriting sorted outputs to: {out_dir.resolve()}")

In [ ]:
# ============================================================
# Step 1: Compute global gene frequency
# (frequency = number of times each gene appears across all rows)
# ============================================================
gene_freq = Counter()
odd_token_rows = 0
total_rows = 0
bad_prefix_rows = 0

for organ_name, src in src_files:
    with src.open() as f:
        data = json.load(f)

    for row in data:
        total_rows += 1
        input_str = prepare_input(row.get("input", ""))

        if not (isinstance(input_str, str) and input_str.startswith(GENES_PREFIX)):
            bad_prefix_rows += 1
            continue

        tokens = input_str[len(GENES_PREFIX):].split()
        if len(tokens) % 2 != 0:
            odd_token_rows += 1

        gene_freq.update(g for g, _ in parse_genes(input_str))

print(f"\nunique genes: {len(gene_freq)}")
print(f"rows with odd gene token count: {odd_token_rows} / {total_rows}")
print(f"rows missing/invalid '{GENES_PREFIX}' prefix: {bad_prefix_rows} / {total_rows}")

In [ ]:
# ============================================================
# Step 2: Rewrite datasets with frequency-sorted genes
# ============================================================
rows_with_no_genes = 0

for organ_name, src in src_files:
    with src.open() as f:
        data = json.load(f)

    sorted_rows = []
    for row in data:
        genes = parse_genes(prepare_input(row.get("input", "")))

        if not genes:
            rows_with_no_genes += 1
            sorted_rows.append(row)  # keep row unchanged if genes can't be parsed
            continue

        # Sort by global frequency desc; ties keep original order (sorted() is stable).
        reordered = sorted(genes, key=lambda gc: -gene_freq[gc[0]])

        new_input = GENES_PREFIX + " ".join(f"{g} {c}" for g, c in reordered)
        new_row = dict(row)            # preserve all fields: instruction/output/label/etc.
        new_row["input"] = new_input
        sorted_rows.append(new_row)

    # Derive the output filename from OUTPUT_NAME_STYLE.
    if OUTPUT_NAME_STYLE == "noorgan":
        part_match = re.search(r"part_(\d+)", src.stem)
        part_num = part_match.group(1) if part_match else "unknown"
        dest = out_dir / f"{organ_name}_celldata_part_{part_num}_sorted_noorgan.json"
    else:  # "stem"
        dest = out_dir / f"{src.stem}_sorted.json"

    with dest.open("w") as f:
        json.dump(sorted_rows, f, ensure_ascii=True, indent=2)

    print(f"wrote {dest.name} ({len(sorted_rows)} rows)")

print(f"\nrows kept unchanged due to unparsable/empty genes: {rows_with_no_genes}")